In [1]:
# Setup

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel
import numpy as np
import pandas as pd
import mlflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score

# ── Spark + MinIO ─────────────────────────────────────────────────
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("HospitalOptimizer-LSTM") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://hospital-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "hospital") \
    .config("spark.hadoop.fs.s3a.secret.key", "hospital123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} démarré")
print(f"✅ TensorFlow {tf.__version__}")

✅ Spark 3.5.0 démarré
✅ TensorFlow 2.15.0


In [2]:
# Chargement et préparation

# ── Charger Gold depuis MinIO ─────────────────────────────────────
df_gold = spark.read.parquet("s3a://gold/parquet/mimic_features.parquet").cache()

# ── Sélectionner les features pour le LSTM ────────────────────────
features = [
    "num_labs", "num_distinct_labs", "num_abnormal_labs",
    "num_rx", "num_distinct_drugs", "complexity_score",
    "abnormal_lab_rate", "LOS"
]

target = "high_risk"

# ── Convertir en Pandas ───────────────────────────────────────────
df_pd = df_gold.select(features + [target]).dropna().toPandas()

print(f"✅ Données chargées : {df_pd.shape[0]:,} lignes")
print(f"   Features  : {len(features)}")
print(f"\nDistribution target (high_risk) :")
print(df_pd[target].value_counts())
print(f"\nPourcentage patients à risque : {df_pd[target].mean()*100:.1f}%")

✅ Données chargées : 122,756 lignes
   Features  : 8

Distribution target (high_risk) :
high_risk
1    95996
0    26760
Name: count, dtype: int64

Pourcentage patients à risque : 78.2%


In [3]:
# Normalisation et split

from sklearn.model_selection import train_test_split

# ── Normalisation ─────────────────────────────────────────────────
scaler = MinMaxScaler()
X = scaler.fit_transform(df_pd[features])
y = df_pd[target].values

# ── Split train/test ──────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Reshape pour LSTM (samples, timesteps, features) ──────────────
X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test  = X_test.reshape(X_test.shape[0],  1, X_test.shape[1])

print(f"✅ Données préparées")
print(f"   Train : {X_train.shape}")
print(f"   Test  : {X_test.shape}")
print(f"   y_train distribution : {y_train.mean()*100:.1f}% positifs")

✅ Données préparées
   Train : (98204, 1, 8)
   Test  : (24552, 1, 8)
   y_train distribution : 78.2% positifs


In [4]:
from tensorflow.keras.callbacks import EarlyStopping

# ── Architecture LSTM ─────────────────────────────────────────────
model_lstm = Sequential([
    LSTM(64, input_shape=(1, len(features)), return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

model_lstm.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 1, 64)             18688     
                                                                 
 dropout (Dropout)           (None, 1, 64)             0         
                                                                 
 lstm_1 (LSTM)               (None, 32)                12416     
                                                                 
 dropout_1 (Dropout)         (None, 32)                0         
                                                                 
 dense (Dense)               (None, 16)                528       
                                                                 
 dense_1 (Dense)             (None, 1)                 17        
                                                                 
Total params: 31649 (123.63 KB)
Trainable params: 31649 

In [5]:
# Entraînement

# ── Gérer le déséquilibre des classes ─────────────────────────────
class_weight = {0: 3.0, 1: 1.0}  # donner plus de poids à la classe minoritaire

# ── Early stopping pour éviter le surapprentissage ────────────────
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# ── Entraînement ──────────────────────────────────────────────────
history = model_lstm.fit(
    X_train, y_train,
    epochs=10,
    batch_size=512,
    validation_split=0.2,
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1
)

print("✅ LSTM entraîné")

Epoch 1/10
154/154 [==============================] - 264s 1s/step - loss: 0.9195 - accuracy: 0.7211 - precision: 0.8143 - recall: 0.8333 - val_loss: 0.5444 - val_accuracy: 0.6356 - val_precision: 0.9224 - val_recall: 0.5837
Epoch 2/10
154/154 [==============================] - 80s 522ms/step - loss: 0.7965 - accuracy: 0.6164 - precision: 0.9300 - recall: 0.5509 - val_loss: 0.5314 - val_accuracy: 0.6156 - val_precision: 0.9357 - val_recall: 0.5467
Epoch 3/10
154/154 [==============================] - 12s 76ms/step - loss: 0.7918 - accuracy: 0.6130 - precision: 0.9345 - recall: 0.5430 - val_loss: 0.5321 - val_accuracy: 0.6158 - val_precision: 0.9349 - val_recall: 0.5474
Epoch 4/10
154/154 [==============================] - 31s 196ms/step - loss: 0.7900 - accuracy: 0.6175 - precision: 0.9350 - recall: 0.5489 - val_loss: 0.5313 - val_accuracy: 0.6172 - val_precision: 0.9347 - val_recall: 0.5495
Epoch 5/10
154/154 [==============================] - 8s 53ms/step - loss: 0.7885 - accuracy: 0

In [6]:
# Évaluation
# ── Prédictions ───────────────────────────────────────────────────
y_pred_proba = model_lstm.predict(X_test)
y_pred       = (y_pred_proba > 0.5).astype(int).flatten()

# ── Métriques ─────────────────────────────────────────────────────
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("=" * 50)
print("RÉSULTATS LSTM")
print("=" * 50)
print(f"Précision : {precision*100:.1f}%  (cible > 85%)")
print(f"Recall    : {recall*100:.1f}%")
print(f"F1 Score  : {f1*100:.1f}%")
print(f"\nKPI : {'✅ ATTEINT' if precision > 0.85 else '❌ NON ATTEINT'}")

768/768 [==============================] - 163s 199ms/step
RÉSULTATS LSTM
Précision : 93.7%  (cible > 85%)
Recall    : 55.6%
F1 Score  : 69.8%

KPI : ✅ ATTEINT


In [8]:
# Logger dans MLflow

mlflow.set_tracking_uri("http://hospital-mlflow:5000")
mlflow.set_experiment("/hospital-optimizer/lstm")

with mlflow.start_run(run_name="lstm_complications_v1"):

    # Paramètres
    mlflow.log_param("epochs",     len(history.history['loss']))
    mlflow.log_param("batch_size", 512)
    mlflow.log_param("lstm_units", "64-32")
    mlflow.log_param("dropout",    0.2)
    mlflow.log_param("features",   len(features))

    # Métriques
    mlflow.log_metric("precision",    round(precision, 3))
    mlflow.log_metric("recall",       round(recall, 3))
    mlflow.log_metric("f1",           round(f1, 3))
    mlflow.log_metric("kpi_achieved", 1 if precision > 0.85 else 0)

    print("✅ Run MLflow enregistré")
    print(f"   Précision : {precision*100:.1f}%")
    print(f"   KPI       : {'✅ ATTEINT' if precision > 0.85 else '❌ NON ATTEINT'}")

# Sauvegarder le modèle localement
model_lstm.save("/home/jovyan/src/models/lstm_model.keras")
print("✅ Modèle sauvegardé localement")

✅ Run MLflow enregistré
   Précision : 93.7%
   KPI       : ✅ ATTEINT
✅ Modèle sauvegardé localement
